# Create Fact Tables
* race results
* qualifying
* sprint results
* pit 

In [0]:
from pyspark.sql.functions import (col,upper,trim,concat_ws,coalesce,lit,udf,when,max,create_map,to_timestamp,date_sub,to_date,first,lower,size,split,initcap,element_at)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql import Row
from pyspark.sql import functions as F


In [0]:
tables=spark.catalog.listTables("f1_warehouse.silver")

temp_gold={}

for table in tables:
    df=spark.table(f"f1_warehouse.silver.{table.name}")
    temp_gold[table.name]=df

In [0]:
for table_names,df in temp_gold.items():
    print(table_names)

In [0]:
tables=spark.catalog.listTables("f1_warehouse.gold")

dim={}

for table in tables:
    df=spark.table(f"f1_warehouse.gold.{table.name}")
    dim[table.name]=df

In [0]:
for table_names,df in dim.items():
    print(table_names)

### Create fact table for race day
* grain: race results of 1 driver per team for a race_meeting_id

In [0]:
k_race_results=temp_gold["kaggle_results"]
of1_race_results=temp_gold["openf1_race_results"]


In [0]:
#combine kaggle sprint and kaggle results

kaggle_sprint=temp_gold["kaggle_sprint_results"].withColumn("event_type",lit("Sprint"))
k_race_results=k_race_results.withColumn("event_type",lit("Race"))
                                                        
print(kaggle_sprint.columns)
print(k_race_results.columns)

In [0]:
common_cols = [
    "result_id",
    "race_id",
    "driver_id",
    "constructor_id",
    "number",
    "grid",
    "position",
    "position_text",
    "position_order",
    "points",
    "laps",
    "time",
    "milliseconds",
    "fastest_lap",
    "fastest_lap_time",
    "status_id",
    "ingestion_timestamp",
    "file_name",
    "event_type"
]

kaggle_sprint=kaggle_sprint.select(common_cols)
k_race_results=k_race_results.select(common_cols)
k_race_results=k_race_results.unionByName(kaggle_sprint)

In [0]:
#join meeting key

k_race_results = (
    k_race_results.join(
        dim["dim_meetings"].select("kaggle_race_id", "race_meeting_id","date_start"),
        k_race_results["race_id"] == dim["dim_meetings"]["kaggle_race_id"],
        "left"
    )
    .drop("kaggle_race_id")
)

of1_race_results = (
    of1_race_results.join(
        dim["dim_meetings"].select("openf1_meeting_id", "race_meeting_id","date_start"),
        of1_race_results["meeting_id"] == dim["dim_meetings"]["openf1_meeting_id"],
        "left"
    )
    .drop("openf1_meeting_id")
)

In [0]:


# order k_race_results by date start
 
k_race_results= k_race_results.orderBy("date_start")


In [0]:
# join on name
#driver dim does not have constuctor id as the team u belong to can continuously change 
of1_drivers=temp_gold["openf1_drivers"].alias("of1")
teams = dim["dim_teams"].select(
    "name",
    "constructor_id"
).alias("t")

of1_drivers = (
    of1_drivers
    .join(
        teams,
        col("of1.team_name") == col("t.name"),
        how="left"
    )
    .drop(col("t.name"))
)

In [0]:
of1_drivers.filter((col("meeting_id") == 1215)& (col("session_id")==9223)).show(100, False)

In [0]:
# error with of1_drivers' code e.g max verstappen code is not ARO

In [0]:
of1_drivers=of1_drivers.select("driver_number","first_name","full_name","last_name","constructor_id","session_id","meeting_id")


In [0]:
dim["dim_driver"].show(5)

of1_drivers.show(5)

In [0]:
of1_drivers.count()

In [0]:
from pyspark.sql.functions import col, split, trim, when, element_at, initcap
#some driver names have null first name and last name in of1 drivers so we split them 

of1_cleaned_driver_names = (
    of1_drivers
    .withColumn(
        "first_name",
        when(
            col("first_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), 1)))
        ).otherwise(col("first_name"))
    )
    .withColumn(
        "last_name",
        when(
            col("last_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), -1)))
        ).otherwise(col("last_name"))
    )
)


In [0]:
#join driver id on first name,last name and number


drivers = dim["dim_driver"].withColumn("number", col("number").try_cast("int"))

of1_cleaned_drivers = (
    of1_cleaned_driver_names.alias("of1")
    .join(
        drivers.alias("d"),
        (col("of1.first_name") == col("d.forename")) &
        (col("of1.last_name") == col("d.surname")) &
        (col("of1.driver_number") == col("d.number")),
        "inner"
    )
    .drop("d.number", "d.surname","d.forename")
)



In [0]:
of1_cleaned_drivers.count()

In [0]:
dropped_drivers=of1_cleaned_driver_names.alias("of1").join(
    drivers.alias("d"),
    (col("of1.first_name") == col("d.forename")) &
    (col("of1.last_name") == col("d.surname")) &
    (col("of1.driver_number") == col("d.number")),
    "left_anti"
)

In [0]:
dropped_drivers.count()

In [0]:
display(dropped_drivers)

these happen to be test drivere/f2 drivers so its okay if they are dropped

In [0]:
of1_race_results.count()

In [0]:
#join name acronym on drivernumber and session id

of1_race_results = (
    of1_race_results.alias("r")
    .join(
        of1_cleaned_drivers.select(
            "session_id",
            "driver_number",
            "constructor_id",
            "driver_id"
        ).alias("d"),
        (col("r.session_id") == col("d.session_id")) &
        (col("r.driver_number") == col("d.driver_number")),
        how="left"
    )
    .drop(col("d.session_id"))
    .drop(col("d.driver_number"))
)

In [0]:
of1_race_results.filter(col("driver_id").isNull()).count()


In [0]:

#join starting grid to openf1

of1_race_results=(
    of1_race_results
    .join( temp_gold["openf1_starting_grid"]
            .select(
                "meeting_id",
                "driver_number",
                col("position").alias("starting_position")),
        on=["meeting_id", "driver_number"],
        how="left"
    )
)


In [0]:
#cast  miliseconds to int


k_race_results = k_race_results.withColumn(
    "milliseconds",
    when(col("milliseconds") == r"\\N", None)
    .otherwise(col("milliseconds"))
    .try_cast("int")
)

In [0]:
#convert miliseconds to seconds
k_race_results=k_race_results.withColumn("duration",col("milliseconds").try_cast("double")/1000)

In [0]:


# Window for each race
race_window = Window.partitionBy("race_meeting_id")

# Get winner's race time (milliseconds)
k_race_results = (
    k_race_results
    .withColumn(
        "winner_time",
        F.min(col("duration")).over(race_window).try_cast("long")
    )
)

# Calculate gap to leader (seconds)
k_race_results = (
    k_race_results
    .withColumn(
        "gap_to_leader",
        when(
            col("duration").isNull(),
            None
        ).otherwise(
            (col("duration").try_cast("long") - col("winner_time").try_cast("long")) / 1000.0
        )
    )
    .drop("winner_time")
)





In [0]:
#create did not finish,did not start and disqualified col flags for kaggle 

k_race_results=(
    k_race_results.join(
        temp_gold["kaggle_status"].select("status_id","status"),
        on=k_race_results["status_id"] == temp_gold["kaggle_status"]["status_id"],
        how="left"))

#derive the flags

k_race_results = (
    k_race_results
    .withColumn(
        "dnf",
        when(
            (~lower(col("status")).contains("finished")) &
            (~lower(col("status")).contains("lap")) &
            (~lower(col("status")).contains("disqualified")),
            True
        ).otherwise(False)
    )
    .withColumn(
        "dns",
        when(lower(col("status")).contains("did not start"), True)
        .otherwise(False)
    )
    .withColumn(
        "dsq",
        when(lower(col("status")).contains("disqualified"), True)
        .otherwise(False)
    )
)


In [0]:
# Before the union, in cell 24 (or right after building k_race_results)
of1_race_results = (of1_race_results.withColumn("gap_to_leader", col("gap_to_leader").try_cast("double")).withColumn("duration", col("duration").try_cast("double")))
k_race_results=k_race_results.withColumn("position",col("position").try_cast("double"))
  

In [0]:
#select relevant cols for each table 

k_race_results=k_race_results.select(
    "race_meeting_id",
    "driver_id",
    "constructor_id",
    col("grid").alias("starting_position"),
    "position",
    "gap_to_leader",
    "duration",
    "points",
    col("laps").alias("number_of_laps"),
    "dnf",
    "dns",
    "dsq"
)

of1_race_results=of1_race_results.select(
    "race_meeting_id",
    "driver_id",
    "constructor_id",
    "starting_position",
    "position",
    "gap_to_leader",
    "duration",
    "points",
    "number_of_laps",
    "dnf",
    "dns",
    "dsq"
)

#union both tables

fact_race_results=k_race_results.unionByName(of1_race_results)

#


In [0]:
fact_race_results.filter(col("race_meeting_id").isNull()).count()

In [0]:
fact_race_results.count()

In [0]:
fact_race_results = fact_race_results.dropDuplicates()

In [0]:
fact_race_results.count()

In [0]:
#add not complete col
fact_race_results=fact_race_results.withColumn("nc",col("position").isNull() & ~(col("dnf") | col("dns") | col("dsq")))

In [0]:
fact_race_results.show(5)

In [0]:
#write to gold table

fact_race_results.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("f1_warehouse.gold.fact_race_results")

### Duplicates

In [0]:
# check if true grain has duplicates
fact_race_results.groupBy("race_meeting_id", "driver_id","constructor_id").count().filter(col("count") > 1).show()

In [0]:
of1_drivers.groupBy(
    "session_id",
    "driver_number"
).count().filter(
    col("count") > 1
).show()

In [0]:
dim["dim_driver"].groupBy(
    "driver_id"
).count().filter(
    col("count") > 1
).show()

In [0]:
dup_keys = (
    fact_race_results
    .groupBy("race_meeting_id", "driver_id")
    .count()
    .filter(col("count") > 1)
)
dup_keys.show()

In [0]:
temp_gold["kaggle_results"].filter(
    (col("race_id") == 540) & (col("driver_id") == 229)
).show(truncate=False)

In [0]:
dup_rows = (
    fact_race_results
    .join(dup_keys.select("race_meeting_id", "driver_id"), on=["race_meeting_id", "driver_id"], how="inner")
    .orderBy("race_meeting_id", "driver_id")
)

dup_rows.show(50, truncate=False)

In [0]:
# Is the kaggle_race_id -> race_meeting_id mapping actually 1:1?
dim["dim_meetings"].groupBy("race_meeting_id").agg(
    F.countDistinct("kaggle_race_id").alias("distinct_kaggle_race_ids")
).filter(col("distinct_kaggle_race_ids") > 1).show()

In [0]:
temp_gold["kaggle_results"].groupBy("race_id", "driver_id").count().filter(col("count") > 1).show()

In [0]:
dim["dim_driver"].groupBy("driver_number").count().filter(col("count") > 1).show()


### Pit Fact Table
* grain:1 row per pit stop made by each driver per race_meeting_id

In [0]:
temp_gold["kaggle_pit_stops"].show(5)

In [0]:
temp_gold["openf1_pit"].show(5)

In [0]:
dim["dim_driver"].show(5)

dim["dim_driver"].printSchema()

In [0]:
of1_pit=temp_gold["openf1_pit"]
k_pit=temp_gold["kaggle_pit_stops"]

In [0]:
of1_drivers=temp_gold["openf1_drivers"]

In [0]:
of1_drivers.count()

In [0]:
of1_drivers=of1_drivers.select("driver_number","first_name","full_name","last_name","session_id","meeting_id")

In [0]:
#join of1 drivers and dim drivers on name acronym

of1_cleaned_driver_names = (
    of1_drivers
    .withColumn(
        "first_name",
        when(
            col("first_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), 1)))
        ).otherwise(col("first_name"))
    )
    .withColumn(
        "last_name",
        when(
            col("last_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), -1)))
        ).otherwise(col("last_name"))
    )
)


In [0]:
#join driver id on first name,last name and number

drivers = dim["dim_driver"].withColumn("number", col("number").try_cast("int"))

of1_cleaned_drivers = (
    of1_cleaned_driver_names.alias("of1")
    .join(
        drivers.alias("d"),
        (col("of1.first_name") == col("d.forename")) &
        (col("of1.last_name") == col("d.surname")) &
        (col("of1.driver_number") == col("d.number")),
        "inner"
    )
    .drop("d.number", "d.surname","d.forename")
)

* use inner join cos u dont need test drivers/reserved drivers pit stop results 

In [0]:
dropped_drivers=of1_cleaned_driver_names.alias("of1").join(
    drivers.alias("d"),
    (col("of1.first_name") == col("d.forename")) &
    (col("of1.last_name") == col("d.surname")) &
    (col("of1.driver_number") == col("d.number")),
    "left_anti"
)

In [0]:
dropped_drivers.show()

In [0]:
of1_cleaned_drivers.count()

In [0]:
#join meeting key

k_pit= (
    k_pit.join(
        dim["dim_meetings"].select("kaggle_race_id", "race_meeting_id"),
        k_pit["race_id"] == dim["dim_meetings"]["kaggle_race_id"],
        "left"
    )
    .drop("kaggle_race_id")
)

of1_pit= (
    of1_pit.join(
        dim["dim_meetings"].select("openf1_meeting_id", "race_meeting_id"),
        of1_pit["meeting_id"] == dim["dim_meetings"]["openf1_meeting_id"],
        "left"
    )
    .drop("openf1_meeting_id")
)

In [0]:
of1_pit.count()

In [0]:

#drop test drivers/reserved drivers
#join name acronym on driver number and session id
of1_pit = (
    of1_pit.alias("p")
    .join(
        of1_cleaned_drivers.select(
            "driver_number",
            "session_id",
            "driver_id",
        ).alias("d"),
        (col("p.session_id") == col("d.session_id")) &
        (col("p.driver_number") == col("d.driver_number")),
        how="inner" #use inner cos we want to drop test drivers and reserved drivers 
    )
    .drop(col("d.session_id"))
    .drop(col("d.driver_number"))
)

In [0]:
unmatched_pit = (
    of1_pit.alias("p")
    .join(
        of1_cleaned_drivers.select(
            "driver_number",
            "session_id",
            "driver_id",
        ).alias("d"),
        (col("p.session_id") == col("d.session_id")) &
        (col("p.driver_number") == col("d.driver_number")),
        how="leftanti"
    )
    .drop(col("d.session_id"))
    .drop(col("d.driver_number")))


In [0]:
of1_pit.count()

In [0]:
k_pit.show(5)

k_pit.printSchema()

In [0]:
k_pit = k_pit.select(
    "driver_id",
    "race_meeting_id",
    col("stop").alias("stop_number"),
    col("lap").alias("lap_number"),
    (col("milliseconds") / 1000).cast("double").alias("stop_duration"),
    col("time").alias("stop_time")
)

of1_pit = of1_pit.select(
    "driver_id",
    "race_meeting_id",
    lit(None).cast("int").alias("stop_number"),
    "lap_number",
    col("pit_duration").cast("double").alias("stop_duration"),
    col("date").alias("stop_time")
)

# Join both
fact_pit_stops = k_pit.unionByName(of1_pit)

In [0]:
#perform checks
print(fact_pit_stops.filter(col("driver_id").isNull()).count())

print(fact_pit_stops.filter(col("race_meeting_id").isNull()).count())


In [0]:
fact_pit_stops.count()

In [0]:
fact_pit_stops=fact_pit_stops.dropDuplicates()

In [0]:
fact_pit_stops.count()

In [0]:
# load to gold

fact_pit_stops.write.format("delta").mode("overwrite").option("OverwriteSchema", "true").saveAsTable("f1_warehouse.gold.fact_pit_stops")

### Create fact table for Qualifying

* grain: 1 driver per 1 race_meeting_id

In [0]:
temp_gold["kaggle_qualifying"].show(5)

In [0]:
temp_gold["openf1_qualifying_results"].show(5)

temp_gold["openf1_pit"].show(5)

In [0]:
k_qualifying = temp_gold["kaggle_qualifying"]

of1_qualifying = temp_gold["openf1_qualifying_results"]

In [0]:
# join on name
#driver dim does not have constuctor id as the team u belong to can continuously change 
of1_drivers=temp_gold["openf1_drivers"].alias("of1")
teams = dim["dim_teams"].select(
    "name",
    "constructor_id"
).alias("t")

of1_drivers = (
    of1_drivers
    .join(
        teams,
        col("of1.team_name") == col("t.name"),
        how="left"
    )
    .drop(col("t.name"))
)

In [0]:
of1_drivers=of1_drivers.select("driver_number","first_name","full_name","last_name","session_id","meeting_id","constructor_id")

In [0]:
of1_drivers.count()

In [0]:
#join of1 drivers and dim drivers on name acronym

of1_cleaned_driver_names = (
    of1_drivers
    .withColumn(
        "first_name",
        when(
            col("first_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), 1)))
        ).otherwise(col("first_name"))
    )
    .withColumn(
        "last_name",
        when(
            col("last_name").isNull(),
            initcap(trim(element_at(split(col("full_name"), " "), -1)))
        ).otherwise(col("last_name"))
    )
)

In [0]:

#join driver id on first name,last name and number

drivers = dim["dim_driver"].withColumn("number", col("number").try_cast("int"))

of1_cleaned_drivers = (
    of1_cleaned_driver_names.alias("of1")
    .join(
        drivers.alias("d"),
        (col("of1.first_name") == col("d.forename")) &
        (col("of1.last_name") == col("d.surname")) &
        (col("of1.driver_number") == col("d.number")),
        "inner"
    )
    .drop("d.number", "d.surname","d.forename")
)

In [0]:
of1_cleaned_drivers.count()

In [0]:
dropped_drivers=of1_cleaned_driver_names.alias("of1").join(
    drivers.alias("d"),
    (col("of1.first_name") == col("d.forename")) &
    (col("of1.last_name") == col("d.surname")) &
    (col("of1.driver_number") == col("d.number")),
    "left_anti"
)

In [0]:
#join constructor id and driver id for openf1

#join driver key 

of1 = of1_qualifying.alias("of1")
of1_cleaned_drivers= of1_cleaned_drivers.select("driver_id", "driver_number", "constructor_id").alias("d")

of1_qualifying = (
    of1.join(of1_cleaned_drivers,
        col("of1.driver_number") == col("d.driver_number"),
        "left"
    )
    .drop(col("d.driver_number"))   # only drops the dimension copy
)



In [0]:
of1_qualifying.show(5)

In [0]:
of1_qualifying.filter(col("driver_id").isNull()).show()


In [0]:
#join meeting key
# split duration into q1,q2,q3 

k_qualifying = (
    k_qualifying.join(
        dim["dim_meetings"].select("kaggle_race_id", "race_meeting_id"),
        k_qualifying["race_id"] == dim["dim_meetings"]["kaggle_race_id"],
        "left"
    )
    .drop("kaggle_race_id")
)

of1_qualifying = (
    of1_qualifying.join(
        dim["dim_meetings"].select("openf1_meeting_id", "race_meeting_id"),
        of1_qualifying["meeting_id"] == dim["dim_meetings"]["openf1_meeting_id"],
        "left"
    )
    .drop("openf1_meeting_id")
)




In [0]:
# check if its 3 items in the array

temp_gold["openf1_qualifying_results"].select(
    "driver_number",
    "duration",
    size(split("duration", ',')).alias("n_times")
).show(truncate=False)

In [0]:
#change string array to spark array
from pyspark.sql.functions import split, regexp_replace, trim, col

of1_qualifying = (
    of1_qualifying
    .withColumn("duration_array",split(regexp_replace(regexp_replace(col("duration"), r"[\[\]]", ""), " ", ""),",")
    )
)

In [0]:
from pyspark.sql.functions import try_element_at, lit

def time_to_seconds(column):
    parts = split(col(column), ":")
    return (
        try_element_at(parts, lit(1)).try_cast("double") * 60
        + try_element_at(parts, lit(2)).try_cast("double")
    )

In [0]:
from pyspark.sql.functions import expr
#split duration into q1,q2,q3
of1_qualifying = of1_qualifying.select(
    "driver_id",
    "race_meeting_id",
    "constructor_id",
    "position",
    expr("try_element_at(duration_array, 1)").try_cast("double").alias("q1"),
    expr("try_element_at(duration_array, 2)").try_cast("double").alias("q2"),
    expr("try_element_at(duration_array, 3)").try_cast("double").alias("q3"),
    "number_of_laps",
)


k_qualifying=k_qualifying.select(
    "driver_id",
    "race_meeting_id",
    "constructor_id",
    "position",
    time_to_seconds("q1").alias("q1"),
    time_to_seconds("q2").alias("q2"),
    time_to_seconds("q3").alias("q3"),
    col("number").alias("number_of_laps")
)


#join both tables

fact_qualifying=k_qualifying.unionByName(of1_qualifying)


In [0]:
# data checks
of1_qualifying.show(5)

In [0]:
fact_qualifying.filter(col("driver_id").isNull()).count()

In [0]:
fact_qualifying.filter(col("driver_id").isNull()).show()

In [0]:
print("Kaggle:", k_qualifying.filter(col("driver_id").isNull()).count())
print("OpenF1:", of1_qualifying.filter(col("driver_id").isNull()).count())
print("Fact:", fact_qualifying.filter(col("driver_id").isNull()).count())

In [0]:
#failed data quality check
fact_qualifying.filter(col("driver_id").isin(866)).show()

In [0]:
fact_qualifying.filter(col("constructor_id").isNull()).show()
fact_qualifying.filter(col("constructor_id").isNull()).groupBy("driver_id").count().filter(col("count")>1).show()
fact_qualifying.filter(col("constructor_id").isNull()).select("race_meeting_id").distinct().show()

In [0]:
fact_qualifying = (
    fact_qualifying.alias("f")
    .join(
        dim["dim_meetings"]
        .select("race_meeting_id", "year")
        .alias("m"),
        col("f.race_meeting_id") == col("m.race_meeting_id"),
        "left"
    )
    .withColumn(
        "constructor_id",
        when(
            col("f.constructor_id").isNull() & (col("m.year") == 2025),
            218
        )
        .when(
            col("f.constructor_id").isNull() & (col("m.year") == 2026),
            216
        )
        .otherwise(col("f.constructor_id"))
    )
    .drop("year")
    .drop(col("m.race_meeting_id")
    
))

In [0]:
fact_qualifying.filter(
    col("driver_id") == 866
).select(
    "driver_id",
    "constructor_id"
).distinct().show()

In [0]:
#save to gold

fact_qualifying.write.format("delta").mode("overwrite").option("OverwriteSchema", "true").saveAsTable("f1_warehouse.gold.fact_qualifying")


### Remaining tasks
* change date start to date end and change it to race date
* add date for qualifying
* fix driver_dim for null driver id
* create udf for data checks 

